<a href="https://colab.research.google.com/github/rikesh28/Credit_Card_Fraud_Detection/blob/main/notebooks/6)%20Production_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score, precision_recall_curve
import pickle

In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Fraud_Detection_System_Project/2) Data/Processed Data/train_df.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Fraud_Detection_System_Project/2) Data/Processed Data/test_df.csv')

In [ ]:
# We are defining only features available in API --> these are the only features we can ask from users in our UI

api_features = [
    'TransactionAmt', 'ProductCD',
    'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain',
    # Engineered features we can create
    'TransactionAmt_log',
    'TransactionAmt_decimal',
    'is_round_amount',
    'email_domain_match',
    'P_email_is_common',
    'R_email_is_common',
    'has_P_email',
    'has_R_email'
]

print(f'Using {len(api_features)} API-Freindly features')

Using 20 API-Freindly features


In [ ]:
X_train = train_df[api_features].copy()
y_train = train_df['isFraud']

X_test = test_df[api_features].copy()
y_test = test_df['isFraud']

In [ ]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Fraud rate (train): {y_train.mean()*100:.2f}%")

X_train shape: (472432, 20)
X_test shape: (118108, 20)
Fraud rate (train): 3.51%


In [ ]:
scale_pos_W = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_W:.2f}")

Scale pos weight: 27.46


In [ ]:
# Traning API-Friendly Model
api_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_W,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    use_label_encoder=False
)
api_model.fit(X_train, y_train, verbose = False)
print('Model Trained')

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:23:28] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model Trained


In [ ]:
# Evaluate
y_pred_proba = api_model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print("\n=== API MODEL PERFORMANCE ===")
print(classification_report(y_test, y_pred))

print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")


=== API MODEL PERFORMANCE ===
              precision    recall  f1-score   support

           0       0.99      0.83      0.90    114044
           1       0.13      0.67      0.21      4064

    accuracy                           0.83    118108
   macro avg       0.56      0.75      0.56    118108
weighted avg       0.96      0.83      0.88    118108


ROC-AUC: 0.8242
Precision: 0.1251
Recall: 0.6732
F1-Score: 0.2110


In [ ]:
# Check legitimate predictions
legit_mask = y_test == 0
legit_probs = y_pred_proba[legit_mask]

print(f"\n=== LEGITIMATE TRANSACTION CHECK ===")
print(f"Mean fraud probability: {legit_probs.mean():.4f}")
print(f"% with >60% fraud prob: {(legit_probs > 0.6).sum() / len(legit_probs) * 100:.2f}%")


=== LEGITIMATE TRANSACTION CHECK ===
Mean fraud probability: 0.2993
% with >60% fraud prob: 9.55%


In [ ]:
# Saving API Model
with open('/content/drive/MyDrive/Colab_Notebooks/Fraud_Detection_System_Project/3) Models/api_model_xgb.pkl','wb') as f:
  pickle.dump(api_model, f)

# Saving Features names
pd.DataFrame({'feature': api_features}).to_csv('/content/drive/MyDrive/Colab_Notebooks/Fraud_Detection_System_Project/3) Models/api_feature_names.csv', index = False)

print("\nAPI model saved as: api_model_xgb.pkl")
print("Features saved as: api_feature_names.csv")


API model saved as: api_model_xgb.pkl
Features saved as: api_feature_names.csv


In [ ]:
#Optimal Threshold Analysis

# Calculate precision and recall at different thresholds
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Calculate F1 scores for each threshold
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

# Find optimal threshold (max F1)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

print(f"Default Threshold: 0.50")
print(f"Optimal Threshold: {optimal_threshold:.3f}")
print(f"Optimal F1 Score: {optimal_f1:.4f}")

# Make predictions with optimal threshold
y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)

print(f"\n=== PERFORMANCE AT DIFFERENT THRESHOLDS ===")
print(f"{'Threshold':<15} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("-" * 60)

for thresh in [0.3, 0.4, 0.5, optimal_threshold, 0.6, 0.7]:
    y_pred_thresh = (y_pred_proba >= thresh).astype(int)
    prec = precision_score(y_test, y_pred_thresh)
    rec = recall_score(y_test, y_pred_thresh)
    f1 = f1_score(y_test, y_pred_thresh)
    marker = " ← Optimal" if abs(thresh - optimal_threshold) < 0.01 else ""
    print(f"{thresh:<15.3f} {prec:<15.4f} {rec:<15.4f} {f1:<15.4f}{marker}")

Default Threshold: 0.50
Optimal Threshold: 0.757
Optimal F1 Score: 0.2974

=== PERFORMANCE AT DIFFERENT THRESHOLDS ===
Threshold       Precision       Recall          F1-Score       
------------------------------------------------------------
0.300           0.0655          0.8617          0.1218         
0.400           0.0874          0.7712          0.1570         
0.500           0.1251          0.6732          0.2110         
0.757           0.2494          0.3684          0.2974          ← Optimal
0.600           0.1656          0.5315          0.2525         
0.700           0.2180          0.4213          0.2873         


# **Summary of the Notebook --> Why do we need this production model although we already have our final tuned model?**

##Problem Statement
The research model (434 features, 0.88 ROC-AUC) cannot be deployed to production because:
- Many features require historical data not available at transaction time
- Complex feature engineering increases latency
- Data pipeline infrastructure too expensive

## Objective
Create a production-ready model with:
- Only real-time available features
- <100ms inference time
- (>0.75) ROC-AUC maintained

## Feature Selection Criteria
✅ Available at transaction time  
✅ No historical lookups  
✅ Simple computation  
❌ V-features (anonymized, unavailable)  
❌ D-features (require transaction history)  
❌ C-features (require aggregations)  

## Results
- Features: 434 → 21 (95% reduction)
- ROC-AUC: 0.88 → 0.78 (10% drop)
- Inference: 500ms → <100ms (5x faster)
- **Conclusion**: Acceptable tradeoff for production deployment
